# DATA255 architecture efficiency benchmark

Run on one GPU runtime for all three models. This follows Nathan's timm benchmark: 14 outputs, FP32, batch size 1, RGB 224x224, four CPU threads, ten warmup iterations, and 50 timed forward passes per trial. Five timing trials are averaged and the standard deviation of trial means is saved.

Each architecture uses a fresh subprocess. No checkpoint, image data, or AUC measurement is needed. Parameters and forward GFLOPs/GMACs are measured once; CPU/GPU latency and inference peak allocated GPU memory are measured in the same runtime. FLOP counts cover the operators registered by PyTorch FlopCounterMode. GMACs use Nathan's FLOPs/2 convention. Inference memory includes model/input/tensor allocations, but excludes CUDA context and allocations outside PyTorch. Training memory belongs in the separate training-results table.

Choose a free T4 GPU runtime and Run All. The final cell downloads the benchmark results.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
if importlib.util.find_spec('timm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
WORK = Path.cwd() / 'data255_efficiency'
WORK.mkdir(exist_ok=True)
SCRIPT = WORK / 'benchmark_efficiency.py'
SCRIPT.write_text('#!/usr/bin/env python3\n"""Nathan\'s architecture benchmark, with repeated timing and inference memory.\n\nNo training data or checkpoints are read. Each architecture uses a fresh process.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport statistics\nimport subprocess\nimport sys\nimport time\n\nMODELS = ["resnet18", "mobilenetv3_small_100", "deit_tiny_patch16_224"]\n\n\ndef hardware(torch, timm):\n    cpu_name = platform.processor()\n    info = Path("/proc/cpuinfo")\n    if info.exists():\n        for line in info.read_text().splitlines():\n            if line.startswith("model name"):\n                cpu_name = line.split(":", 1)[1].strip()\n                break\n    quota = Path("/sys/fs/cgroup/cpu.max")\n    return {\n        "platform": platform.platform(), "python": sys.version,\n        "cpu_name": cpu_name, "logical_cpu_count": os.cpu_count(),\n        "cpu_affinity_count": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,\n        "cgroup_cpu_max": quota.read_text().strip() if quota.exists() else None,\n        "torch_threads": torch.get_num_threads(),\n        "torch_interop_threads": torch.get_num_interop_threads(),\n        "torch": str(torch.__version__), "timm": timm.__version__,\n        "cuda_runtime": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),\n        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n        "gpu_total_memory_mb": torch.cuda.get_device_properties(0).total_memory / 2**20\n        if torch.cuda.is_available() else None,\n    }\n\n\ndef timed_forward(model, x, torch, args, cuda=False):\n    samples = []\n    with torch.no_grad():\n        for _ in range(args.warmup):\n            model(x)\n        if cuda:\n            torch.cuda.synchronize()\n            baseline = torch.cuda.memory_allocated()\n            torch.cuda.reset_peak_memory_stats()\n        else:\n            baseline = None\n        for _ in range(args.trials):\n            if cuda:\n                torch.cuda.synchronize()\n            start = time.perf_counter()\n            for _ in range(args.runs):\n                model(x)\n            if cuda:\n                torch.cuda.synchronize()\n            samples.append((time.perf_counter() - start) / args.runs * 1000)\n    result = {"mean_ms": statistics.mean(samples),\n              "std_ms": statistics.stdev(samples) if len(samples) > 1 else 0.0,\n              "trial_mean_ms": samples}\n    if cuda:\n        result.update(\n            resident_mb=baseline / 2**20,\n            peak_allocated_mb=torch.cuda.max_memory_allocated() / 2**20,\n            incremental_peak_mb=(torch.cuda.max_memory_allocated() - baseline) / 2**20,\n            peak_reserved_mb=torch.cuda.max_memory_reserved() / 2**20,\n        )\n    return result\n\n\ndef worker(args):\n    import torch\n    import timm\n    from torch.utils.flop_counter import FlopCounterMode\n\n    torch.set_num_threads(args.threads)\n    torch.manual_seed(42)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    if args.require_cuda and not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required: select a GPU runtime before running this benchmark.")\n    model = timm.create_model(args.worker, pretrained=False, num_classes=14).eval()\n    x = torch.randn(1, 3, 224, 224)\n    params = sum(p.numel() for p in model.parameters())\n    with torch.no_grad():\n        assert tuple(model(x).shape) == (1, 14)\n        # Profile explicit attention matmuls because the fused CPU SDPA path can\n        # be absent from FlopCounterMode\'s registered formulas. Restore the\n        # default fused path before latency/memory measurement.\n        attention = [(m, m.fused_attn) for m in model.modules() if hasattr(m, "fused_attn")]\n        for module, _ in attention:\n            module.fused_attn = False\n        try:\n            with FlopCounterMode(display=False) as counter:\n                model(x)\n        finally:\n            for module, enabled in attention:\n                module.fused_attn = enabled\n    flops = int(counter.get_total_flops())\n    assert flops > 0\n    breakdown = {str(k): int(v) for k, v in counter.get_flop_counts().get("Global", {}).items()}\n    cpu = timed_forward(model, x, torch, args)\n    del model, x\n    gc.collect()\n    gpu = None\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        torch.manual_seed(42)\n        model = timm.create_model(args.worker, pretrained=False, num_classes=14).eval().cuda()\n        x = torch.randn(1, 3, 224, 224, device="cuda")\n        gpu = timed_forward(model, x, torch, args, cuda=True)\n        del model, x\n        gc.collect()\n        torch.cuda.empty_cache()\n    result = {\n        "model": args.worker, "parameter_count": params, "params_m": params / 1e6,\n        "forward_flops": flops, "gflops": flops / 1e9, "gmacs": flops / 2e9,\n        "cpu_latency_ms": cpu["mean_ms"], "cpu_latency_std_ms": cpu["std_ms"],\n        "gpu_latency_ms": gpu["mean_ms"] if gpu else None,\n        "gpu_latency_std_ms": gpu["std_ms"] if gpu else None,\n        "gpu_peak_allocated_mb": gpu["peak_allocated_mb"] if gpu else None,\n        "gpu_resident_mb": gpu["resident_mb"] if gpu else None,\n        "gpu_incremental_peak_mb": gpu["incremental_peak_mb"] if gpu else None,\n        "batch_size": 1, "input_shape": "1x3x224x224", "num_classes": 14,\n        "dtype": "float32", "cpu_threads": args.threads,\n        "warmup_iters": args.warmup, "timed_iters_per_trial": args.runs,\n        "trials": args.trials,\n    }\n    details = {"summary": result, "cpu_timing": cpu, "gpu_timing": gpu,\n               "counted_flop_operators": breakdown, "flop_profile_attention": "explicit matmul; default fused path restored for timing", "hardware": hardware(torch, timm)}\n    (args.output_dir / f"{args.worker}.json").write_text(json.dumps(details, indent=2))\n    print(json.dumps(result), flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--output-dir", type=Path, default=Path("benchmark_results"))\n    p.add_argument("--threads", type=int, default=4)\n    p.add_argument("--warmup", type=int, default=10)\n    p.add_argument("--runs", type=int, default=50)\n    p.add_argument("--trials", type=int, default=5)\n    p.add_argument("--require-cuda", action="store_true")\n    p.add_argument("--worker", choices=MODELS)\n    args = p.parse_args()\n    if min(args.threads, args.warmup, args.runs, args.trials) < 1:\n        p.error("threads, warmup, runs, and trials must be positive")\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    if args.worker:\n        worker(args)\n        return\n    results = []\n    for name in MODELS:\n        print(f"Benchmarking {name}: CPU and GPU, batch size 1", flush=True)\n        cmd = [sys.executable, "-u", str(Path(__file__).resolve()), "--worker", name,\n               "--output-dir", str(args.output_dir.resolve()), "--threads", str(args.threads),\n               "--warmup", str(args.warmup), "--runs", str(args.runs), "--trials", str(args.trials)]\n        if args.require_cuda:\n            cmd.append("--require-cuda")\n        subprocess.run(cmd, check=True)\n        details = json.loads((args.output_dir / f"{name}.json").read_text())\n        results.append(details["summary"])\n        with (args.output_dir / "efficiency_benchmark.csv").open("w", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=list(results[0]))\n            writer.writeheader()\n            writer.writerows(results)\n    (args.output_dir / "benchmark_protocol.json").write_text(json.dumps({\n        "models": MODELS, "initialization": "random; pretrained=False; no checkpoint",\n        "scope": "architecture-level, model-forward-only inference; preprocessing and data transfer excluded",\n        "execution": "timm eager eval, torch.no_grad, FP32, TF32 disabled, cudnn.benchmark=False",\n        "input_shape": [1, 3, 224, 224], "num_classes": 14,\n        "cpu_threads": args.threads, "warmup_iters": args.warmup,\n        "timed_iters_per_trial": args.runs, "trials": args.trials,\n        "latency_statistic": "mean of trial means; standard deviation across trial means",\n        "gpu_timing": "perf_counter; CUDA synchronization before and after each timed block",\n        "complexity": "PyTorch FlopCounterMode supported-operator estimates; GFLOPs=FLOPs/1e9; GMACs=FLOPs/2e9",\n        "memory": "PyTorch peak allocated MiB after warmup and peak reset; includes model/input/inference tensors; excludes CUDA context and non-PyTorch allocations",\n        "isolation": "fresh subprocess per architecture; CPU model deleted before GPU model allocation",\n    }, indent=2))\n    print("Benchmark complete:", args.output_dir / "efficiency_benchmark.csv", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
print('Prepared', SCRIPT)


In [ ]:
command = [sys.executable, '-u', str(SCRIPT), '--output-dir', str(WORK / 'results'), '--require-cuda']
p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
assert p.wait() == 0, 'Benchmark failed; inspect the output above.'


In [ ]:
import pandas as pd
display(pd.read_csv(WORK / 'results/efficiency_benchmark.csv'))


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/data255_efficiency_results', 'zip', WORK / 'results')
print('Results:', archive)
files.download(archive)
